# ORBIT memory system first-slice showcase

This notebook showcases ORBIT's current first real memory-system slice.

It demonstrates:
- turn-end memory capture into session + durable memory layers
- derived embedding persistence
- hybrid retrieval (`semantic + lexical`)
- retrieval weight biasing
- application-vs-postgres backend compare view
- memory probe snapshots as context artifacts
- notebook inspection helpers for the memory surface
- reusable memory-notebook companion scaffolding
- top-level summary metadata for current memory posture
- condensed status signals for human-readable readiness
- dataframe-first status rendering for notebook ergonomics


This surface is now intended to behave like a small workbench slice rather than a one-off demo: bundle first, status card second, then drill down into records/probes/artifacts.

In [1]:
import sys
from pathlib import Path

ROOT = Path('/Volumes/2TB/MAS/openclaw-core/ORBIT').resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from orbit.notebook import (
    build_durable_bias_service,
    capture_memory_showcase_turns,
    create_memory_showcase_bundle,
    memory_compare_backends_dataframe,
    memory_context_artifacts_dataframe,
    memory_probe_dataframe,
    memory_showcase_summary_frames,
    memory_status_summary_frame,
    session_messages_dataframe,
)


## 1. Create the reusable showcase bundle

This now comes from companion helpers rather than ad-hoc notebook-only setup.

In [2]:
bundle = create_memory_showcase_bundle()
store = bundle['store']
service = bundle['service']
session = bundle['session']
bundle['db_path']

PosixPath('/var/folders/s7/s1_f2dw13yx1zwfqd5vdgqkr0000gn/T/orbit_memory_showcase_864l9idt/orbit.db')

## 2. Capture the reusable default showcase turns


In [3]:
captured = capture_memory_showcase_turns(store=store, service=service, session=session)
[(record.scope, record.memory_type, record.summary_text) for record in captured]

[('session',
  'summary',
  'User: I prefer concise ORBIT architecture answers and remember to ship the transcript store. | Assistant: Decision: the plan is to keep transcript and memory separated.'),
 ('durable',
  'user_preference',
  'I prefer concise ORBIT architecture answers and remember to ship the transcript store'),
 ('durable',
  'todo',
  'I prefer concise ORBIT architecture answers and remember to ship the transcript store'),
 ('durable',
  'decision',
  'the plan is to keep transcript and memory separated'),
 ('session',
  'summary',
  'User: I also like memory probes that stay inspectable in the notebook. | Assistant: Lesson: keep retrieval results visible as auxiliary context rather than hidden transcript rewrites.'),
 ('durable',
  'lesson',
  'keep retrieval results visible as auxiliary context rather than hidden transcript rewrites')]

## 3. Inspect transcript truth separately from memory truth


In [4]:
session_messages_dataframe(store, session.session_id)

,message_id,session_id,turn_index,role,content,created_at,provider_message_id,metadata
0,msg_6199d9a0bf9b,session_memory_showcase,1,user,I prefer concise ORBIT architecture answers an...,2026-04-06 15:12:30.295095+00:00,None,{}
1,msg_edb2a17063fa,session_memory_showcase,2,assistant,Decision: the plan is to keep transcript and m...,2026-04-06 15:12:30.295188+00:00,None,{}
2,msg_e78f80de173e,session_memory_showcase,3,user,I also like memory probes that stay inspectabl...,2026-04-06 15:12:30.295203+00:00,None,{}
3,msg_381a137235c7,session_memory_showcase,4,assistant,Lesson: keep retrieval results visible as auxi...,2026-04-06 15:12:30.295220+00:00,None,{}


## 4. Pull the standard memory bundle

This returns a structured summary plus notebook-facing DataFrames for drill-down.

In [5]:
query_text = 'what are my concise orbit memory decisions and todos?'
frames = memory_showcase_summary_frames(
    store=store,
    service=service,
    session=session,
    query_text=query_text,
)
frames.keys()

dict_keys(['summary', 'records', 'scope_summary', 'embeddings', 'probe', 'compare', 'artifacts'])

In [6]:
frames['summary']

{'session_id': 'session_memory_showcase',
 'run_id': 'run_memory_showcase',
 'query_text': 'what are my concise orbit memory decisions and todos?',
 'record_count': 6,
 'session_record_count': 2,
 'durable_record_count': 4,
 'embedding_count': 6,
 'probe_result_count': 6,
 'compare_row_count': 6,
 'probe_artifact_count': 3,
 'memory_types': ['decision', 'lesson', 'summary', 'todo', 'user_preference'],
 'embedding_models': ['demo-memory-showcase'],
 'dominant_memory_types': ['summary', 'lesson', 'decision'],
 'has_durable_memory': True,
 'has_probe_artifacts': True,
 'has_embeddings': True,
 'retrieval_readiness': 'ready',
 'current_backend_posture': {'default_backend': 'application',
  'default_strategy': 'hybrid_embedding_lexical_v1',
  'compare_supports_postgres_stub': True,
  'postgres_execution_enabled': False},
 'status': {'memory_layer': 'active',
  'retrieval_layer': 'inspectable',
  'artifact_layer': 'captured',
  'backend_mode': 'application',
  'backend_strategy': 'hybrid_emb

In [7]:
memory_status_summary_frame(frames['summary'])

,session_id,run_id,record_count,durable_record_count,embedding_count,probe_result_count,probe_artifact_count,retrieval_readiness,has_durable_memory,has_embeddings,has_probe_artifacts,dominant_memory_types,embedding_models,backend_mode,backend_strategy,postgres_mode,memory_layer,retrieval_layer,artifact_layer
0,session_memory_showcase,run_memory_showcase,6,4,6,6,3,ready,True,True,True,"summary, lesson, decision",demo-memory-showcase,application,hybrid_embedding_lexical_v1,stub_compare_only,active,inspectable,captured


In [ ]:
frames['records']

In [ ]:
frames['scope_summary']

In [ ]:
frames['embeddings']

In [ ]:
frames['probe']

## 5. Bias ranking toward durable memory and compare


In [ ]:
durable_biased_service = build_durable_bias_service(store=store)
default_probe = memory_probe_dataframe(
    store,
    session_id=session.session_id,
    query_text=query_text,
    limit=10,
    scope='all',
    memory_service=service,
)
durable_probe = memory_probe_dataframe(
    store,
    session_id=session.session_id,
    query_text=query_text,
    limit=10,
    scope='all',
    memory_service=durable_biased_service,
)
{'default': default_probe.head(5), 'durable_biased': durable_probe.head(5)}

## 6. Compare application vs postgres backend plans

At this stage, postgres remains an explicit compare/stub posture rather than an enabled execution path.

In [ ]:
memory_compare_backends_dataframe(
    store,
    session_id=session.session_id,
    query_text=query_text,
    limit=10,
    scope='all',
    memory_service=service,
)

## 7. Inspect probe snapshots as artifacts


In [ ]:
memory_context_artifacts_dataframe(store, session.conversation_id, artifact_type='memory_probe_snapshot')

## 8. Takeaways

This notebook now behaves like a small memory workbench slice: build a bundle, inspect the rendered status card, then drill down into records, probes, compare views, and artifacts.
